# NB00 · Datos: contrato, perfilado y decisiones de negocio

Evidencia real para las decisiones **D01-D04**. La validación de contrato completa y la longitud en tokens con el tokenizer del modelo elegido quedan para cuando ese modelo esté decidido (NB03).

---

### 📐 Convención de corpus — qué se mide sobre qué

Toda cabecera de sección lleva marcado su corpus, y ninguna celda mezcla los dos:

| Marca | Corpus | Fichero | Para qué |
|---|---|---|---|
| 🔬 **MUESTRA** | 1.500 registros | `catalogo_muestra.csv` | Desarrollo y calibración (condición 3 del plan) |
| 📚 **COMPLETO** | 15.000 registros | `catalogo_productos.csv` | Confirmación de las decisiones antes de fijarlas |
| ⚪ **SIN CATÁLOGO** | — | juicios y consultas | Evidencia que no depende del catálogo |

⚠️ Importa para el README: las cifras de una sección 🔬 **no** son las del catálogo completo. Por ejemplo, `brand` tiene 2,93 % de nulos en la muestra y 4,4 % en los 15.000 registros.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd

from aurum.datos import (
    brand_normalization_collisions,
    esci_label_counts_per_query,
    null_field_rates,
    qrels_coverage_in_catalog,
    qrels_pool_sizes,
    text_field_label_summary,
    value_frequency,
)

DATA = Path("..") / "data"

# Los dos corpus, cargados con nombres que no se pueden confundir.
muestra = pd.read_csv(DATA / "catalogo_muestra.csv")
completo = pd.read_csv(DATA / "catalogo_productos.csv")

consultas = pd.read_csv(DATA / "consultas_desarrollo.csv")
relevancias = pd.read_csv(DATA / "relevancias_desarrollo.csv")

print(f"🔬 MUESTRA : {len(muestra):>6} registros, {muestra['record_id'].nunique():>6} record_id únicos")
print(f"📚 COMPLETO: {len(completo):>6} registros, {completo['record_id'].nunique():>6} record_id únicos")
print(f"consultas_desarrollo: {len(consultas)} consultas · "
      f"relevancias_desarrollo: {len(relevancias)} juicios")


🔬 MUESTRA :   1500 fichas,   1500 record_id únicos
📚 COMPLETO:  15000 fichas,  15000 record_id únicos
consultas_desarrollo: 8 consultas · relevancias_desarrollo: 248 juicios


## 🔬 Distribución de valores — `brand` y `color` (top 15 por frecuencia)

In [2]:
value_frequency(muestra, "brand").head(15)


,brand,n_filas,pct_filas
0,(vacío),44,2.93
1,Litoral,15,1.00
2,Lenovo,15,1.00
3,ASUS,10,0.67
4,find.,9,0.60
5,LG,9,0.60
6,HOGAR24,8,0.53
7,HP,7,0.47
8,PUMA,7,0.47
9,DUÉRMETE ONLINE,6,0.40


In [3]:
value_frequency(muestra, "color").head(15)


,color,n_filas,pct_filas
0,(vacío),549,36.60
1,Negro,151,10.07
2,Blanco,77,5.13
3,Multicolor,31,2.07
4,Gris,27,1.80
5,Rojo,22,1.47
6,Azul,18,1.20
7,Marrón,17,1.13
8,Verde,17,1.13
9,Black,16,1.07


## D01 · ⚪ ¿Qué cuenta como relevante para Recall@10 / MRR@10? (E vs E+S)

Se calcula solo sobre `relevancias_desarrollo.csv`: no depende de qué catálogo se use, así que la cifra es idéntica en muestra y en completo.

In [4]:
esci_counts = esci_label_counts_per_query(relevancias).merge(
    consultas[["query_id", "query_text"]], on="query_id"
)
esci_counts[["query_id", "query_text", "E", "S", "C", "I", "relevantes_solo_E", "relevantes_E_mas_S"]]


,query_id,query_text,E,S,C,I,relevantes_solo_E,relevantes_E_mas_S
0,13357,base tapizada 160x200 sin patas,4,27,0,9,4,31
1,18868,botines marrones mujer tacon medio,7,2,0,7,7,9
2,28703,convertibles 2 en 1 portátil tactil,25,14,0,1,25,39
3,31224,cámaras bridge baratas,5,10,1,0,5,15
4,33633,disfraz halloween talla grande hombre,1,3,2,10,1,4
5,38249,estantes sin taladro habitacion,30,5,1,4,30,35
6,43240,funda ipad air 4 sin tapa,24,11,0,5,24,35
7,61533,lentejas sin gluten,25,5,0,10,25,30


## D02 · 🔬 Nulos en los campos que pueden entrar en el texto codificado

In [5]:
null_field_rates(muestra, ["title", "brand", "color", "text"])


,campo,n_nulos,n_filas,pct_nulos
0,title,0,1500,0.00
1,brand,44,1500,2.93
2,color,549,1500,36.60
3,text,0,1500,0.00


### 🔬 ¿El `text` ya trae las etiquetas Marca/Color (ES o EN)? — evidencia por regex

In [6]:
text_field_label_summary(muestra)


,campo,n_filas,n_con_etiqueta_en_text,pct_con_etiqueta_en_text,n_etiqueta_con_campo_vacio
0,marca,1500,1456,97.07,0
1,color,1500,1001,66.73,50


### D02 (confirmación) · 📚 Los mismos nulos sobre el catálogo completo

La política de nulos se aplicará al codificar los 15.000 registros, no solo a la muestra: conviene saber si las proporciones se sostienen a escala real.

In [7]:
null_field_rates(completo, ["title", "brand", "color", "text"])


,campo,n_nulos,n_filas,pct_nulos
0,title,0,15000,0.00
1,brand,658,15000,4.39
2,color,5609,15000,37.39
3,text,0,15000,0.00


In [8]:
text_field_label_summary(completo)


,campo,n_filas,n_con_etiqueta_en_text,pct_con_etiqueta_en_text,n_etiqueta_con_campo_vacio
0,marca,15000,14342,95.61,0
1,color,15000,9834,65.56,443


## D03 · 🔬 Colisiones al normalizar `brand`

In [9]:
n_distinct_raw = muestra["brand"].dropna().nunique()
resumen = []
for mode in ["raw", "casefold", "unaccent"]:
    colisiones = brand_normalization_collisions(muestra, mode)
    resumen.append({
        "modo": mode,
        "marcas_distintas_crudas": n_distinct_raw,
        "grupos_con_colision": len(colisiones),
        "marcas_fusionadas": int(colisiones["n_marcas_crudas"].sum()) if len(colisiones) else 0,
    })
pd.DataFrame(resumen)


,modo,marcas_distintas_crudas,grupos_con_colision,marcas_fusionadas
0,raw,1191,0,0
1,casefold,1191,0,0
2,unaccent,1191,0,0


In [10]:
brand_normalization_collisions(muestra, "casefold").sort_values("n_marcas_crudas", ascending=False).head(20)


,normalizada,marcas_crudas,n_marcas_crudas


### D03 (confirmación) · 📚 El mismo análisis sobre el catálogo completo

Sobre la muestra no salió ninguna colisión en ningún modo: no prueba nada, ni a favor ni en contra. Se repite aquí sobre los 15.000 registros para confirmar la decisión con evidencia real.

In [11]:
print(f"📚 COMPLETO: {len(completo)} registros, "
      f"{completo['brand'].dropna().nunique()} marcas distintas")


📚 COMPLETO: 15000 fichas, 9054 marcas distintas


In [12]:
n_distinct_raw_completo = completo["brand"].dropna().nunique()
resumen_completo = []
for mode in ["raw", "casefold", "unaccent"]:
    colisiones = brand_normalization_collisions(completo, mode)
    resumen_completo.append({
        "modo": mode,
        "marcas_distintas_crudas": n_distinct_raw_completo,
        "grupos_con_colision": len(colisiones),
        "marcas_fusionadas": int(colisiones["n_marcas_crudas"].sum()) if len(colisiones) else 0,
    })
pd.DataFrame(resumen_completo)


,modo,marcas_distintas_crudas,grupos_con_colision,marcas_fusionadas
0,raw,9054,0,0
1,casefold,9054,66,133
2,unaccent,9054,77,156


In [13]:
brand_normalization_collisions(completo, "casefold").sort_values("n_marcas_crudas", ascending=False).head(20)


,normalizada,marcas_crudas,n_marcas_crudas
43,rc ocio,"[Rc Ocio, Rc ocio, RC ocio]",3
0,3 pommes,"[3 pommes, 3 Pommes]",2
1,actecom,"[ACTECOM, actecom]",2
2,alfa,"[Alfa, ALFA]",2
4,angkorly,"[Angkorly, ANGKORLY]",2
3,anaya infantil y juvenil,"[Anaya Infantil y Juvenil, ANAYA INFANTIL Y JU...",2
6,blomus,"[Blomus, blomus]",2
7,buffalo,"[BUFFALO, Buffalo]",2
8,bugatti,"[Bugatti, bugatti]",2
5,awesafe,"[AWESAFE, awesafe]",2


📚 Grupos que solo aparecen al quitar acentos (no detectados por `casefold`):

In [14]:
cf_normalizadas = set(brand_normalization_collisions(completo, "casefold")["normalizada"])
ua = brand_normalization_collisions(completo, "unaccent")
ua[~ua["normalizada"].isin(cf_normalizadas)]


,normalizada,marcas_crudas,n_marcas_crudas
5,aposan,"[Aposán, Aposan]",2
7,ayudas dinamicas,"[AYUDAS DINAMICAS, Ayudas Dinámicas]",2
20,generico,"[Genérico, Generico]",2
29,l'oreal paris,"[L'Oréal Paris, L'Oreal Paris]",2
30,l'oreal paris elvive,"[L'Oréal Paris Elvive, L'Oreal Paris Elvive]",2
31,l'oreal paris make-up designer,"[L'Oreal Paris Make-up Designer, L'Oréal Paris...",2
32,la jolie muse,"[La Jolíe Muse, LA JOLIE MUSE]",2
39,muhle,"[MÜHLE, Muhle]",2
42,nescafe,"[NESCAFÉ, Nescafé]",2
43,nestle,"[Nestlé, Nestle]",2


## D04 · ⚪ Tamaño del pool juzgado por consulta, frente al catálogo completo

El tamaño del pool sale de los juicios, no del catálogo. La columna `catalogo_completo` es el universo real de búsqueda de la ejecución final.

In [15]:
pool = qrels_pool_sizes(relevancias).merge(consultas[["query_id", "query_text"]], on="query_id")
pool["catalogo_completo"] = len(completo)
pool[["query_id", "query_text", "pool_size", "catalogo_completo"]]


,query_id,query_text,pool_size,catalogo_completo
0,13357,base tapizada 160x200 sin patas,40,15000
1,18868,botines marrones mujer tacon medio,16,15000
2,28703,convertibles 2 en 1 portátil tactil,40,15000
3,31224,cámaras bridge baratas,16,15000
4,33633,disfraz halloween talla grande hombre,16,15000
5,38249,estantes sin taladro habitacion,40,15000
6,43240,funda ipad air 4 sin tapa,40,15000
7,61533,lentejas sin gluten,40,15000


### Precondición de D04 · 🔬 ¿los `product_id` juzgados existen en la muestra?

Si no están, Recall@10/nDCG de desarrollo calculados buscando sobre la muestra no son fiables sin importar qué universo de puntuación elijamos en D04.

In [16]:
qrels_coverage_in_catalog(relevancias, muestra).merge(
    consultas[["query_id", "query_text"]], on="query_id"
)[["query_id", "query_text", "n_juzgados", "n_presentes", "pct_presentes"]]


,query_id,query_text,n_juzgados,n_presentes,pct_presentes
0,13357,base tapizada 160x200 sin patas,40,40,100.0
1,18868,botines marrones mujer tacon medio,16,16,100.0
2,28703,convertibles 2 en 1 portátil tactil,40,40,100.0
3,31224,cámaras bridge baratas,16,16,100.0
4,33633,disfraz halloween talla grande hombre,16,16,100.0
5,38249,estantes sin taladro habitacion,40,40,100.0
6,43240,funda ipad air 4 sin tapa,40,40,100.0
7,61533,lentejas sin gluten,40,40,100.0


### Precondición de D04 (confirmación) · 📚 los mismos juicios sobre el catálogo completo

In [17]:
qrels_coverage_in_catalog(relevancias, completo).merge(
    consultas[["query_id", "query_text"]], on="query_id"
)[["query_id", "query_text", "n_juzgados", "n_presentes", "pct_presentes"]]


,query_id,query_text,n_juzgados,n_presentes,pct_presentes
0,13357,base tapizada 160x200 sin patas,40,40,100.0
1,18868,botines marrones mujer tacon medio,16,16,100.0
2,28703,convertibles 2 en 1 portátil tactil,40,40,100.0
3,31224,cámaras bridge baratas,16,16,100.0
4,33633,disfraz halloween talla grande hombre,16,16,100.0
5,38249,estantes sin taladro habitacion,40,40,100.0
6,43240,funda ipad air 4 sin tapa,40,40,100.0
7,61533,lentejas sin gluten,40,40,100.0
